# トークン重要度 / attention 分析

## この実験の目的

Diffusion Plannerは、自車の履歴だけでなく、周辺車両、車線、経路、道路境界など多数の入力を「トークン」として受け取る。この実験では、**モデルがどの入力を実際に利用して予測しているか**を調べる。

主に次の問いに答えることが目的である。

1. 周辺車両や車線を取り除くと、予測精度はどれくらい悪化するか
2. 最大320台・140車線すべてが必要か、それとも近いものだけで十分か
3. モデル内部のattentionは、どの種類のトークンを強く参照しているか
4. 遠方のトークンが役立っているか、逆にノイズになっていないか

この結果は、入力トークン数の削減やモデル高速化の候補を探すための手がかりになる。ただし、ここで測るのは**現在のcheckpointの使い方**であり、入力を減らしたモデルの最終性能を保証するものではない。最終判断には、入力構成を変更した再学習と閉ループ評価が必要である。

## 実行方法

`run_token_analysis.sh` が分析からHTML作成までを一括実行する。

```bash
MODEL_DIR=best_models/20260730/best_model \
DATADIR=/path/to/dataset \
N_SAMPLES=1024 BATCH_SIZE=64 DEVICE=cuda \
./run_token_analysis.sh
```

処理内容:

1. `scripts/token_importance.py`で入力除去実験を行い、FDE/ADEをTSVへ保存
2. `scripts/attention_analysis.py`でattentionを集計し、JSONへ保存
3. このNotebookを実行し、実行済み`.ipynb`と画像埋め込み済みの単一HTMLを生成

HTMLは画像とスタイルを内部に埋め込むため、元のTSVやJSONがない別のPCでも単体で閲覧できる。

## Feature importance（入力ablation）の方法

ここでいうfeature importanceは、一般的な木モデルのimportanceではなく、**入力を意図的に隠したときに予測誤差がどれだけ変わるか**を測るablation方式である。

### 計算手順

1. 移動距離が`MOVE_MIN_M`以上の評価シーンを選ぶ
2. 入力を変更せず推論し、baselineのFDE/ADEを計算する
3. 同じシーンに対し、対象入力だけをゼロへ置換して再度推論する
4. `ablationの誤差 − baselineの誤差`をimportance（Δ）として記録する

同じシーン、同じモデル、同じ初期trajectoryを使うため、Δは入力変更による差を直接比較しやすい。

### Drop（入力クラス単位）

`drop:neighbors`や`drop:lanes`は、対象クラス全体をゼロへ置き換える。

- neighbors/mapsなどの可変長入力はpadding maskによりattention対象から除外される
- goal pose/turn indicatorsは固定長トークンなので、トークン自体の削除ではなく入力情報を一定値へ置換する

### Top-K（近いものだけ残す）

`nbr_top:16`なら、自車に近い周辺車両16台だけを残す。Kを増やして誤差がbaselineへ近づく地点が、現在のモデルに必要なトークン数の候補になる。ただし「近い」の定義は現在位置までのユークリッド距離であり、将来衝突可能性や道路接続関係による順位ではない。

### Within radius（距離制限）

`nbr_within:50`なら50 m以内の周辺車両だけを残す。Top-Kが台数制限を調べるのに対し、withinは物理的な検知範囲を調べる。

### Δの読み方

- **Δ > 0**: 除去すると悪化した。その入力情報をcheckpointが有効利用している可能性が高い
- **Δ ≈ 0**: 除去しても開ループ位置誤差はほぼ変わらない。この評価条件では寄与が確認できない
- **Δ < 0**: 除去すると改善した。入力がノイズになっている、モデルが誤用している、または標本誤差の可能性がある

Δがゼロでも「その入力が本質的に不要」とは断定できない。安全性に関係する少数シーンへの効果が平均FDEに現れない場合や、入力をゼロにすることで学習分布外になる場合がある。削減判断ではサンプル数を増やして再現性を確認し、衝突率などの閉ループ指標と、入力を削ったモデルの再学習結果も確認する。

## 評価指標と実行パラメータ

### FDE / ADE

- **FDE (Final Displacement Error)**: 予測の最終地点と正解の最終地点の距離。小さいほどよい
- **ADE (Average Displacement Error)**: 全時刻における位置誤差の平均。小さいほどよい
- **Δ FDE / Δ ADE**: ablation結果からbaselineを引いた値。正なら入力除去で悪化、負なら入力除去で改善

### Shellから指定できる主なパラメータ

| 環境変数 | 既定値 | 意味 |
|---|---:|---|
| `MODEL_DIR` | `best_models/20260730/best_model` | `args.json`と`best_model.pth`を含むモデルディレクトリ |
| `DATADIR` | mini dataset | 評価データのルート |
| `VALID_LIST` | `$DATADIR/path_list_valid.json` | 評価対象NPZの一覧。存在しない場合はDATADIR以下の全NPZから生成 |
| `N_SAMPLES` | 128 | 評価する移動シーンの最大数。多いほど安定するが時間がかかる |
| `BATCH_SIZE` | 32 | 一度に推論するサンプル数。大きいほど高速になりやすいがGPUメモリを多く使う |
| `DEVICE` | `cuda` | `cuda`はGPU、`cpu`はCPUで実行 |
| `MOVE_MIN_M` | 5.0 | 予測期間中にこの距離以上移動するシーンだけを評価 |
| `TURN_DEG` | 15.0 | 最終地点の向きがこの角度以上なら旋回シーンとして分類 |
| `OUT_DIR` | dataset名から自動生成 | TSV、JSON、Notebook、HTMLの保存先 |

最初は`N_SAMPLES=128`で動作確認し、本評価では512〜1024以上に増やすとよい。サンプル数を変えた結果同士を比較するときは、同じデータリストと閾値を使用する。

In [ ]:
import os
from pathlib import Path

# Shell実行時は環境変数から成果物を受け取る。対話実行時は既定値を使用。
IMPORTANCE_TSV = Path(
    os.environ.get(
        "TOKEN_IMPORTANCE_TSV", "../best_models/20260730/eval/token_importance_n1024.tsv"
    )
)
ATTENTION_JSON = Path(
    os.environ.get(
        "TOKEN_ATTENTION_JSON", "../best_models/20260730/eval/attention_analysis_n1024.json"
    )
)
RUN_PARAMETERS = {
    "model": os.environ.get("TOKEN_REPORT_MODEL_DIR", "interactive/default"),
    "valid_list": os.environ.get("TOKEN_REPORT_VALID_LIST", "interactive/default"),
    "n_samples": os.environ.get("TOKEN_REPORT_N_SAMPLES", "unknown"),
    "batch_size": os.environ.get("TOKEN_REPORT_BATCH_SIZE", "unknown"),
    "device": os.environ.get("TOKEN_REPORT_DEVICE", "unknown"),
    "move_min_m": os.environ.get("TOKEN_REPORT_MOVE_MIN_M", "unknown"),
    "turn_deg": os.environ.get("TOKEN_REPORT_TURN_DEG", "unknown"),
}

print("run parameters:")
for key, value in RUN_PARAMETERS.items():
    print(f"  {key:>11}: {value}")
print()
print(
    "importance:",
    IMPORTANCE_TSV,
    "->",
    "OK" if IMPORTANCE_TSV.exists() else "NOT FOUND (パスを設定してください)",
)
print("attention :", ATTENTION_JSON, "->", "OK" if ATTENTION_JSON.exists() else "NOT FOUND (任意)")

In [ ]:
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

# 検証済みパレット(dataviz reference)。系列色はエンティティ固定
CAT = {"neighbors": "#2a78d6", "lanes": "#eb6834", "line_strings": "#1baf7a", "route": "#eda100"}
C_WORSE = "#eb6834"  # 劣化(+Δ)
C_BETTER = "#2a78d6"  # 改善(−Δ)
INK = "#1a1a19"
MUTED = "#6b6a63"
GRID = "#e5e4df"


def style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.title.set_color(INK)
    return ax


rows = None
if IMPORTANCE_TSV.exists():
    with open(IMPORTANCE_TSV) as f:
        rows = [
            {k: (v if k == "config" else float(v)) for k, v in r.items()}
            for r in csv.DictReader(f, delimiter="\t")
        ]
    base = next(r for r in rows if r["config"] == "baseline")
    print(
        f"{len(rows)} configs, baseline fde_top={base['fde_top']:.2f}m ade_top={base['ade_top']:.2f}m"
    )

### Feature importanceの全数値

以下の表はTSVに保存された全設定・全指標を省略せず表示する。`top`はモデルが選んだtrajectory、`min`は候補中で正解に最も近いtrajectoryの誤差である。単一trajectory出力ではtopとminは同じ値になる。各`Δ`は同じ指標のbaselineとの差である。

In [ ]:
if rows:
    print(
        f"{'config':<22} {'FDE':>7} {'ΔFDE':>7} {'ADE':>7} {'ΔADE':>7} "
        f"{'minFDE':>8} {'ΔminFDE':>9} {'minADE':>8} {'ΔminADE':>9}"
    )
    for r in rows:
        print(
            f"{r['config']:<22} {r['fde_top']:>7.2f} {r['d_fde_top']:>+7.2f} "
            f"{r['ade_top']:>7.2f} {r['d_ade_top']:>+7.2f} "
            f"{r['min_fde']:>8.2f} {r['d_min_fde']:>+9.2f} "
            f"{r['min_ade']:>8.2f} {r['d_min_ade']:>+9.2f}"
        )
    top_equals_min = all(
        np.isclose(r["fde_top"], r["min_fde"]) and np.isclose(r["ade_top"], r["min_ade"])
        for r in rows
    )
    if top_equals_min:
        print("\n注: この結果ではtopとminが全設定で同値（単一trajectory出力）です。")

## 1. 入力クラス別のFeature importance

左右の横棒は、入力クラス全体を隠したときのΔ FDEとΔ ADEである。FDEは最終地点、ADEは軌跡全体の平均的なずれを見るため、両方を確認する。

- 右向き（正）で長いほど、除去時の悪化が大きく、そのクラスへの依存が強い
- 0付近は、このデータと指標では明確な影響が観測されなかったことを示す
- 左向き（負）は除去後に改善したことを示すが、直ちに「削除すべき」とは判断しない
- Δ FDEだけ大きい場合は終端到達点、Δ ADEだけ大きい場合は途中経路への影響が強い可能性がある

クラス間を比較するときは棒の符号だけでなく絶対値を見る。小さな差はサンプル構成で変動し得るため、異なるサンプル数やsplitでも同じ傾向になるかを確認する。goal pose/turn indicatorsは固定長トークンなので、完全なトークン除去ではなく入力情報の定数化として解釈する。

In [ ]:
if rows:
    drops = [r for r in rows if r["config"].startswith("drop:")]
    drops.sort(key=lambda r: r["d_fde_top"])
    names = [r["config"][5:] for r in drops]

    fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * len(drops) + 1.3), sharey=True)
    for ax, metric, label in (
        (axes[0], "d_fde_top", "FDE"),
        (axes[1], "d_ade_top", "ADE"),
    ):
        vals = [r[metric] for r in drops]
        colors = [C_WORSE if v > 0 else C_BETTER for v in vals]
        ax.barh(names, vals, color=colors, height=0.62)
        ax.axvline(0, color=MUTED, lw=1)
        for i, v in enumerate(vals):
            ax.text(
                v + (0.03 if v >= 0 else -0.03),
                i,
                f"{v:+.2f}",
                va="center",
                ha="left" if v >= 0 else "right",
                fontsize=9,
                color=INK,
            )
        ax.set_xlabel(f"Δ top-mode {label} [m]", color=MUTED)
        ax.set_title(f"{label} importance (baseline {base[label.lower() + '_top']:.2f} m)")
        ax.grid(axis="x", color=GRID, lw=0.6)
        ax.set_axisbelow(True)
        style(ax)
    plt.tight_layout()
    plt.show()

## 2. 近い順Top-Kと距離制限

横軸Kは残したトークン数、上段はFDE、下段はADE、破線は全トークンを使うbaselineである。

### 曲線の読み方

- 小さいKで誤差が高く、Kを増やすと下がる: 追加トークンが予測に役立っている
- あるK以降でFDEとADEの両方がbaseline付近に安定する: そのKが必要数の候補
- Kを増やすと悪化する: 遠いトークンがノイズになっている可能性がある
- FDEとADEの合流点が異なる: 終端到達点と途中経路で必要なトークン数が異なる可能性がある
- 曲線が上下に不規則: サンプル不足、距離順位の不適切さ、分布外入力の影響が考えられる

「baselineと全く同じ点」だけでなく、許容できるΔ FDEとΔ ADEを事前に決めて最小Kを選ぶ。Top-Kはスロット数削減、`*_within`はセンサー・前処理の距離制限を検討するための別の実験であり、同じ結論になるとは限らない。

In [ ]:
if rows:
    sweeps = {
        "nbr_top": ("neighbors", 320),
        "lane_top": ("lanes", 140),
        "ls_top": ("line_strings", 60),
    }
    metrics = [("fde_top", "FDE"), ("ade_top", "ADE")]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey=False)
    for ri, (metric, metric_label) in enumerate(metrics):
        for ax, (prefix, (cls, slots)) in zip(axes[ri], sweeps.items()):
            pts = sorted(
                (
                    (int(r["config"].split(":")[1]), r[metric])
                    for r in rows
                    if r["config"].startswith(prefix + ":")
                ),
            )
            ks = [k for k, _ in pts] + [slots]
            ys = [y for _, y in pts] + [base[metric]]
            ax.plot(ks, ys, marker="o", ms=5, lw=2, color=CAT[cls])
            ax.axhline(base[metric], color=MUTED, lw=1, ls="--")
            ax.text(ks[0], base[metric], " baseline", fontsize=8, color=MUTED, va="bottom")
            ax.set_xscale("log", base=2)
            ax.set_xticks(ks[:-1] + [slots])
            ax.set_xticklabels([str(k) for k in ks[:-1]] + [f"all\n({slots})"], fontsize=8)
            ax.set_title(f"{cls}: keep nearest K", fontsize=10)
            ax.set_xlabel("K (log scale)", color=MUTED)
            ax.grid(axis="y", color=GRID, lw=0.6)
            ax.set_axisbelow(True)
            style(ax)
        axes[ri, 0].set_ylabel(f"top-mode {metric_label} [m]", color=MUTED)
    plt.tight_layout()
    plt.show()

    within = [r for r in rows if "_within:" in r["config"]]
    if within:
        print("距離カットオフ:")
        for r in within:
            print(
                f"  {r['config']:<16} "
                f"FDE={r['fde_top']:6.2f} (Δ={r['d_fde_top']:+.2f})  "
                f"ADE={r['ade_top']:6.2f} (Δ={r['d_ade_top']:+.2f})"
            )

## 3. Attention分析の方法と読み方

### Attentionとは

Fusion Encoderでは、各トークン（query）が他のトークン（key）をどの程度参照するかをattention重みで表す。各queryについて、有効keyに対する重みの合計は1になる。この分析では全headを平均した重みを使う。

Attentionは「どこを参照したか」を示すが、「その入力が最終予測を何m変えたか」を直接示すものではない。そのためattention単独で重要性を断定せず、Feature importanceと併せて解釈する。

### このNotebookで表示する値

- **count share**: 全有効トークンのうち、そのクラスが占める数の割合
- **ego-query attention share**: 自車トークンが各クラスへ向けたattentionの合計。自車予測の文脈として何を参照したかを見る
- **all-query attention share**: 全有効queryから各クラスが受け取ったattentionの平均。Fusion全体での参照傾向を見る
- **selectivity**: `attention share ÷ count share`。トークン数が多いだけでshareが増える影響を補正する
- **value-weighted share**: attention重みにvalueベクトルの大きさを掛けた近似値。重みは大きいが伝える信号が小さいトークンを区別する補助指標。ただしhead構造と出力射影を省略した近似である

### Selectivityの解釈

- **1.0付近**: トークン数にほぼ比例して参照
- **1.0より大きい**: 数の割合以上に選好
- **1.0より小さい**: 数の割合に対して参照が少ない

selectivityが高くてもcount shareが極端に小さければ、attention総量は小さい場合がある。逆にselectivityが1未満でも、トークン数が多いため大きなattention総量を受ける場合がある。倍率とshareの両方を見る。

### 層別・距離別・旋回別の解釈

- 層が深くなるにつれてshareが増えるクラスは、後段で統合される情報である可能性がある
- 距離ビンは、そのクラスに向けられたattentionのうち各距離帯が占める割合。有効トークンが存在するシーンだけで平均する
- 距離ビンはトークン数で補正していないため、遠方ビンのshareが大きくても「1トークンあたり強く注目」とは限らない
- turning/straightのroute share差は、旋回条件でroute参照が変わるかを見る記述的比較。シーン数が少ない側の値は不安定になりやすい

### Feature importanceとの組み合わせ

| Attention | Ablation時の悪化 | 解釈の候補 |
|---|---|---|
| 高い | 大きい | 強く参照し、予測結果にも影響している |
| 高い | 小さい | 参照しているが冗長、または最終出力への寄与が小さい |
| 低い | 大きい | 少量でも重要な情報、別層・別queryで効いている可能性 |
| 低い | 小さい | 現checkpoint・評価条件では利用が限定的 |

この表は診断の出発点であり、因果関係の確定ではない。入力削減の候補は、ablationの再現性、Top-K曲線、距離制限、閉ループ安全指標をまとめて判断する。

In [ ]:
attn = None
if ATTENTION_JSON.exists():
    attn = json.loads(ATTENTION_JSON.read_text())
    print(f"n_samples={attn['n_samples']} (turning {attn['n_turning']}), layers={attn['n_layers']}")

if attn:
    classes = [
        c for c in attn["classes"] if attn["count_share"].get(c, 0) > 0 and c not in ("static",)
    ]
    sel = {c: attn["all_share_avg"][c] / attn["count_share"][c] for c in classes}
    print(
        f"{'class':<16}{'count':>9}{'ego avg':>11}{'all avg':>11}"
        f"{'value-wtd':>12}{'selectivity':>13}"
    )
    for c in attn["classes"]:
        count = attn["count_share"][c]
        ego_avg = float(np.mean(attn["ego_share_per_layer"][c]))
        all_avg = attn["all_share_avg"][c]
        value_avg = attn["vw_share_avg"][c]
        selectivity = all_avg / count if count > 0 else float("nan")
        print(
            f"{c:<16}{100 * count:>8.2f}%{100 * ego_avg:>10.2f}%"
            f"{100 * all_avg:>10.2f}%{100 * value_avg:>11.2f}%{selectivity:>12.2f}x"
        )
    print()
    order = sorted(classes, key=lambda c: sel[c])

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(order) + 1))
    vals = [sel[c] for c in order]
    colors = [C_WORSE if v > 1 else C_BETTER for v in vals]
    ax.barh(order, vals, color=colors, height=0.62)
    ax.axvline(1.0, color=MUTED, lw=1)
    ax.text(1.0, len(order) - 0.2, " 1.0 = count-proportional (dilution)", fontsize=8, color=MUTED)
    for i, v in enumerate(vals):
        ax.text(v + 0.05, i, f"{v:.2f}x", va="center", fontsize=9, color=INK)
    ax.set_xlabel("selectivity (all-query attention share / token count share)", color=MUTED)
    ax.set_title("Attention selectivity by token class")
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    layer_labels = [f"L{i}" for i in range(attn["n_layers"])]
    print(f"{'class':<16}" + "".join(f"{label:>9}" for label in layer_labels))
    for c in attn["classes"]:
        layer_values = attn["ego_share_per_layer"][c]
        print(f"{c:<16}" + "".join(f"{100 * v:>8.2f}%" for v in layer_values))
    print()
    key_classes = [
        c for c in ("neighbors", "lanes", "line_strings", "route") if c in attn["classes"]
    ]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    L = attn["n_layers"]
    for c in key_classes:
        ys = [100 * v for v in attn["ego_share_per_layer"][c]]
        ax.plot(range(L), ys, marker="o", ms=4, lw=2, color=CAT[c])
        ax.text(L - 1 + 0.08, ys[-1], c, fontsize=9, color=CAT[c], va="center")
    ax.set_xticks(range(L))
    ax.set_xticklabels([f"L{i}" for i in range(L)])
    ax.set_xlim(-0.3, L + 1.3)
    ax.set_xlabel("fusion layer", color=MUTED)
    ax.set_ylabel("ego-query attention share [%]", color=MUTED)
    ax.set_title("Ego-token attention share per fusion layer")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

    ax = axes[0]
    route_groups = [
        ("turning", attn["route_share_turning"]),
        ("straight", attn["route_share_straight"]),
    ]
    route_groups = [(label, value) for label, value in route_groups if value is not None]
    labels = [label for label, _ in route_groups]
    vals = [100 * value for _, value in route_groups]
    print("route share:", "  ".join(f"{label}={value:.2f}%" for label, value in zip(labels, vals)))
    ax.bar(labels, vals, color=CAT["route"], width=0.5)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.15, f"{v:.1f}%", ha="center", fontsize=10, color=INK)
    ax.set_ylabel("route attention share [%]", color=MUTED)
    ax.set_title("Route share: turning vs straight (ego query)")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)

    ax = axes[1]
    bins = attn["dist_bins"]
    print("distance-bin share:")
    for label, lane_value, nbr_value in zip(bins, attn["lane_bin_share"], attn["nbr_bin_share"]):
        print(f"  {label:<10} lanes={100 * lane_value:6.2f}%  neighbors={100 * nbr_value:6.2f}%")
    x = np.arange(len(bins))
    w = 0.38
    ax.bar(
        x - w / 2, [100 * v for v in attn["lane_bin_share"]], w, color=CAT["lanes"], label="lanes"
    )
    ax.bar(
        x + w / 2,
        [100 * v for v in attn["nbr_bin_share"]],
        w,
        color=CAT["neighbors"],
        label="neighbors",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(bins, fontsize=9)
    ax.set_ylabel("within-class attention share [%]", color=MUTED)
    ax.set_title("Attention by distance bin (ego query)")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

## 4. 有効トークン数とスロット使用率

Attention分析時のpadding maskから、実際に有効だったトークン数を集計する。この情報は`attention_n*.json`の`token_occupancy`に保存されているため、生データの再走査は行わない。集計対象はattention分析と同じ、移動条件を満たした評価サンプルである。

- **slots**: モデルが確保している最大スロット数
- **mean**: 1シーン当たりの平均有効トークン数
- **p50**: 半数のシーンがこの数以下
- **p95 / p99**: 95% / 99%のシーンがこの数以下
- **max**: 今回評価したシーンで観測された最大値
- **平均使用率**: `mean ÷ slots × 100`

容量削減の初期候補にはp95またはp99を使える。例えばp99 capacityは、今回の標本の99%で有効数を収容できる最小整数である。ただし、残り1%を単純に切り捨てて安全とは限らない。重要な少数シーンが上位1%に含まれる可能性があるため、Feature importanceのTop-K曲線、切り捨て対象の選択規則、閉ループ評価と合わせて判断する。

また、使用率は「スロットが埋まっている割合」であり、トークンの有用性ではない。使用率が低くても少数の遠方トークンが重要な場合があり、使用率が高くても冗長な場合がある。

In [ ]:
occupancy = attn.get("token_occupancy") if attn else None
if occupancy:
    by_class = occupancy["by_class"]
    names = list(by_class)
    print(
        f"{'class':<16}{'slots':>7}{'mean':>9}{'p50':>7}{'p95':>7}"
        f"{'p99':>7}{'max':>7}{'mean util':>12}"
    )
    for name in names:
        s = by_class[name]
        print(
            f"{name:<16}{s['slots']:>7}{s['mean']:>9.1f}{s['p50']:>7.1f}"
            f"{s['p95']:>7.1f}{s['p99']:>7.1f}{s['max']:>7}"
            f"{s['mean_utilization_pct']:>11.1f}%"
        )
    total = occupancy["total"]
    print(
        f"\ntotal: mean={total['mean']:.1f}/{total['slots']} "
        f"({total['mean_utilization_pct']:.1f}%), "
        f"p95={total['p95']:.1f}, p99={total['p99']:.1f}, max={total['max']}"
    )

    variable_names = [n for n in names if by_class[n]["slots"] > 1]
    x = np.arange(len(variable_names))
    width = 0.2
    fig, ax = plt.subplots(figsize=(9, 4.2))
    series = [
        ("mean", "mean", "#2a78d6"),
        ("p95", "p95", "#1baf7a"),
        ("p99", "p99", "#eda100"),
        ("max", "max", "#eb6834"),
    ]
    for i, (label, key, color) in enumerate(series):
        values = [100 * by_class[n][key] / by_class[n]["slots"] for n in variable_names]
        ax.bar(x + (i - 1.5) * width, values, width, label=label, color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(variable_names, rotation=20, ha="right")
    ax.set_ylabel("slot utilization [%]", color=MUTED)
    ax.set_title("Valid-token occupancy by class")
    ax.axhline(100, color=MUTED, lw=1, ls="--")
    ax.legend(frameon=False, ncol=4)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()
else:
    print("token_occupancy is not present; rerun attention_analysis.py with the updated script.")

## 5. 結果を判断するときのチェックリスト

1. baselineのFDE/ADEが想定範囲にあり、評価対象数が十分か
2. class dropのΔは、符号だけでなく絶対値も実用上意味のある大きさか
3. 同じ傾向がサンプル数やデータsplitを変えても再現するか
4. Top-K曲線はどのKで安定し、距離制限の結果とも整合するか
5. 平均・p95・p99・最大の有効数を比較し、少数の高密度シーンを見落としていないか
6. attention shareをトークン数だけで説明できないか、selectivityも確認したか
7. ego-query、all-query、value-weighted shareで大きく異なる場合、その理由を検討したか
8. attentionの傾向とablationの誤差変化が一致するか。一致しない場合を無理に重要／不要と断定していないか
9. 平均FDE/ADEに出ない安全上重要な少数シーンを、個別可視化や閉ループ評価で確認したか

このNotebookから直接言えるのは、選択したデータにおける**現在のcheckpointの入力依存性、attention傾向、有効トークン占有率**である。「モデル一般に不要な入力」や「削除後も安全」という結論には、再学習・複数split・閉ループ評価が追加で必要になる。